# Laboratorio 03 â€” CTEs y Window Functions SQL sobre tu propio dataset

**Semana:** 03 | **Actividad de referencia:** Actividad 03  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica CTEs (`WITH`), subconsultas anidadas y funciones de ventana (`ROW_NUMBER`, `RANK`, `LAG`, `LEAD`, `SUM OVER`) de la Actividad 03 sobre tu dataset personal en Delta.

## Parte 1 â€” DescripciÃ³n del dataset

1. **Nombre, fuente y URL** del dataset.
2. **Columna temporal:** Â¿Tiene alguna columna de fecha/timestamp? Â¿De quÃ© tipo?
3. **Columna de particiÃ³n:** Â¿QuÃ© columna categÃ³rica usarÃ¡s como `PARTITION BY` en las window functions? Â¿Por quÃ©?
4. **Columna de orden:** Â¿QuÃ© columna usarÃ¡s para `ORDER BY` dentro de la ventana?
5. **Preguntas de negocio** que respondan con ranking o anÃ¡lisis de tendencia.

**Escribe tu respuesta aquÃ­:**

## Parte 2 â€” Cargar el dataset como tabla Delta

In [ ]:
VOL          = "/Volumes/workspace/default/week_3"
ARCHIVO      = "tu_archivo.csv"
TABLA        = "workspace.default.lab03_03_mi_dataset"

df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(f"{VOL}/{ARCHIVO}")

df.write.format("delta").mode("overwrite").saveAsTable(TABLA)
print(f"âœ“ Tabla: {TABLA} â€” {df.count():,} filas x {len(df.columns)} columnas")
df.printSchema()

## Parte 3 â€” Perfil tÃ©cnico del dataset

In [ ]:
# EstadÃ­sticas descriptivas bÃ¡sicas
spark.sql(f"SELECT * FROM {TABLA} LIMIT 5").show(truncate=False)
spark.sql(f"SELECT COUNT(*) AS total, COUNT(DISTINCT columna_particion) AS grupos FROM {TABLA}").show()

In [ ]:
# DistribuciÃ³n de registros por columna de particiÃ³n
spark.sql(f"""
    SELECT columna_particion, COUNT(*) AS registros
    FROM {TABLA}
    GROUP BY columna_particion
    ORDER BY registros DESC
    LIMIT 20
""").show(truncate=False)

**ObservaciÃ³n:** Â¿La particiÃ³n estÃ¡ balanceada? Â¿Hay grupos con muy pocos registros donde el ranking pierde sentido?

## Parte 4 â€” CTEs (WITH)

Aplica al menos 2 consultas que usen CTEs para organizar lÃ³gica reutilizable.

In [ ]:
# CTE 1: Preprocesamiento bÃ¡sico + consulta sobre el resultado del CTE
spark.sql(f"""
    WITH base_limpia AS (
        SELECT *
        FROM {TABLA}
        WHERE columna_clave IS NOT NULL
          AND columna_numerica > 0
    ),
    resumen_por_grupo AS (
        SELECT
            columna_particion,
            COUNT(*)           AS total,
            AVG(columna_numerica) AS promedio,
            MAX(columna_numerica) AS maximo
        FROM base_limpia
        GROUP BY columna_particion
    )
    SELECT *
    FROM resumen_por_grupo
    WHERE total > 5
    ORDER BY promedio DESC
    LIMIT 15
""").show(truncate=False)

**Por quÃ© CTEs aquÃ­:** Â¿QuÃ© ventaja tiene usar CTEs en lugar de subconsultas anidadas? Â¿CÃ³mo mejorarÃ­a la legibilidad si aÃ±adieras un tercer CTE?

In [ ]:
# CTE 2: Consulta analÃ­tica libre â€” diseÃ±a un pipeline SQL de 2+ pasos con CTEs
spark.sql(f"""
    WITH paso_1 AS (
        -- Escribe tu primer paso de transformaciÃ³n
        SELECT *
        FROM {TABLA}
    ),
    paso_2 AS (
        -- Aplica lÃ³gica sobre paso_1
        SELECT *
        FROM paso_1
    )
    SELECT * FROM paso_2 LIMIT 20
""").show(truncate=False)

**ConclusiÃ³n CTE 2:**

## Parte 5 â€” Window Functions

In [ ]:
# ROW_NUMBER: asignar un nÃºmero secuencial Ãºnico dentro de cada particiÃ³n
spark.sql(f"""
    SELECT
        columna_particion,
        columna_orden,
        columna_numerica,
        ROW_NUMBER() OVER (
            PARTITION BY columna_particion
            ORDER BY columna_orden DESC
        ) AS row_num
    FROM {TABLA}
    ORDER BY columna_particion, row_num
    LIMIT 30
""").show(truncate=False)

**AnÃ¡lisis:** Filtra con `WHERE row_num = 1` en una subconsulta o CTE. Â¿QuÃ© obtiene ese filtro en tu dataset?

In [ ]:
# Top 1 por grupo usando ROW_NUMBER en CTE
spark.sql(f"""
    WITH ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY columna_particion
                ORDER BY columna_numerica DESC
            ) AS row_num
        FROM {TABLA}
    )
    SELECT *
    FROM ranked
    WHERE row_num = 1
    ORDER BY columna_particion
""").show(truncate=False)

In [ ]:
# RANK vs DENSE_RANK: Â¿quÃ© diferencia hay cuando hay empates?
spark.sql(f"""
    SELECT
        columna_particion,
        columna_numerica,
        RANK()       OVER (PARTITION BY columna_particion ORDER BY columna_numerica DESC) AS rank_con_salto,
        DENSE_RANK() OVER (PARTITION BY columna_particion ORDER BY columna_numerica DESC) AS rank_denso
    FROM {TABLA}
    LIMIT 30
""").show(truncate=False)

**Diferencia RANK vs DENSE_RANK:** En tu dataset, Â¿hay empates? Â¿CuÃ¡ndo usarÃ­as cada uno?

In [ ]:
# LAG y LEAD: acceder al valor anterior/siguiente dentro de la particiÃ³n
spark.sql(f"""
    SELECT
        columna_particion,
        columna_orden,
        columna_numerica,
        LAG(columna_numerica,  1, 0) OVER (
            PARTITION BY columna_particion ORDER BY columna_orden
        ) AS valor_anterior,
        LEAD(columna_numerica, 1, 0) OVER (
            PARTITION BY columna_particion ORDER BY columna_orden
        ) AS valor_siguiente,
        ROUND(
            (columna_numerica - LAG(columna_numerica, 1, 0) OVER (
                PARTITION BY columna_particion ORDER BY columna_orden
            )) * 100.0 / NULLIF(LAG(columna_numerica, 1, 0) OVER (
                PARTITION BY columna_particion ORDER BY columna_orden
            ), 0), 2
        ) AS variacion_pct
    FROM {TABLA}
    ORDER BY columna_particion, columna_orden
    LIMIT 30
""").show(truncate=False)

**AnÃ¡lisis LAG/LEAD:** Â¿QuÃ© filas tienen `variacion_pct` mÃ¡s alta o baja? Â¿QuÃ© evento de negocio podrÃ­a explicar ese cambio?

In [ ]:
# SUM OVER: acumulado corrido por particiÃ³n
spark.sql(f"""
    SELECT
        columna_particion,
        columna_orden,
        columna_numerica,
        SUM(columna_numerica) OVER (
            PARTITION BY columna_particion
            ORDER BY columna_orden
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS acumulado
    FROM {TABLA}
    ORDER BY columna_particion, columna_orden
    LIMIT 30
""").show(truncate=False)

**AnÃ¡lisis acumulado:** Â¿A quÃ© porcentaje del total llega cada registro? Â¿Hay algÃºn punto de inflexiÃ³n donde el acumulado crece mÃ¡s rÃ¡pido?

## Parte 6 â€” Preguntas de negocio

Responde las preguntas de la Parte 1 usando CTEs y window functions.

In [ ]:
# Pregunta 1:
spark.sql(f"""

""").show(truncate=False)

**ConclusiÃ³n pregunta 1:**

In [ ]:
# Pregunta 2:
spark.sql(f"""

""").show(truncate=False)

**ConclusiÃ³n pregunta 2:**

In [ ]:
# Pregunta 3 (la mÃ¡s compleja â€” combina CTEs y al menos 2 window functions):
spark.sql(f"""

""").show(truncate=False)

**ConclusiÃ³n pregunta 3:**

## Parte 7 â€” ReflexiÃ³n final

1. Â¿CuÃ¡ndo es imprescindible usar una window function en lugar de un GROUP BY?
2. Â¿CuÃ¡l es la diferencia entre `ROWS BETWEEN` y `RANGE BETWEEN` en una ventana deslizante?
3. Â¿CÃ³mo expresarÃ­as `LAG(...) OVER (PARTITION BY ... ORDER BY ...)` en PySpark? Â¿La sintaxis es mÃ¡s o menos intuitiva?
4. Â¿QuÃ© patrÃ³n de consulta de este laboratorio reusarÃ­as en un pipeline de producciÃ³n?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_03/laboratorios/lab_03_sql_avanzado.ipynb semana_03/laboratorios/<tu-nombre>/lab_03_sql_avanzado.ipynb

git add semana_03/laboratorios/<tu-nombre>/lab_03_sql_avanzado.ipynb
git commit -m "lab: semana03 lab03 CTEs window functions <nombre-dataset> - <tu-nombre>"
git push origin develop
```